# 03 – Preprocessing & Feature Engineering

**Colab:** [Notebook in Colab öffnen](https://colab.research.google.com/github/jspldrch/DataMining_Final-Project/blob/main/notebooks/03_preprocessing.ipynb) → **Runtime: CPU** → alle Zellen ausführen.

- **Code:** `git pull` nach `/content/DataMining_Final-Project`
- **CSVs:** nur auf Drive unter `MyDrive/DataMining/DataMining_Final-Project/data/`
- **Modus `full`:** Streaming pro Region (~2–4 GB RAM, ~20–40 Min.)

Output für Notebook 04: `outputs/processed/train_labeled.parquet` + `test_features.parquet` (auf Drive)

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from google.colab import drive
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

REPO_URL = "https://github.com/jspldrch/DataMining_Final-Project.git"
REPO_DIR = Path("/content/DataMining_Final-Project")

if IS_COLAB:
    drive.mount("/content/drive")
    if (REPO_DIR / ".git").exists():
        subprocess.run(["git", "pull", "origin", "main"], cwd=REPO_DIR, check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")],
        check=True,
    )
    os.chdir(REPO_DIR)
    sys.path.insert(0, str(REPO_DIR))
    from scripts.colab_setup import init_environment

    env = init_environment(install_deps=False, skip_mount=True, skip_git=True)
else:
    REPO_DIR = Path.cwd()
    if not (REPO_DIR / "scripts").exists() and (REPO_DIR.parent / "scripts").exists():
        REPO_DIR = REPO_DIR.parent
    os.chdir(REPO_DIR)
    sys.path.insert(0, str(REPO_DIR))
    from scripts.colab_setup import init_environment

    env = init_environment(install_deps=True, skip_mount=True, skip_git=True)

PROJECT_ROOT = env["project_root"]
DATA_DIR = env["data_dir"]
TRAIN_PATH = env["train_path"]
TEST_PATH = env["test_path"]
PROCESSED_DIR = env["processed_dir"]
MODE = env["mode"]

_spec = importlib.util.spec_from_file_location(
    "dm_features", PROJECT_ROOT / "scripts" / "features.py"
)
_features = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_features)
build_features = _features.build_features
feature_columns = _features.feature_columns
add_persistence_baseline = _features.add_persistence_baseline

print("Setup OK | MODE:", MODE)

In [ ]:
path_train_std = PROCESSED_DIR / "train_labeled.parquet"
path_test_std = PROCESSED_DIR / "test_features.parquet"
feat_cols = feature_columns()

if MODE == "full":
    _p = PROJECT_ROOT / "scripts" / "preprocess_streaming.py"
    _spec = importlib.util.spec_from_file_location("preprocess_streaming", _p)
    _ps = importlib.util.module_from_spec(_spec)
    _spec.loader.exec_module(_ps)

    print("Streaming (pro Region) — dauert ~20–40 Min. …")
    stats = _ps.preprocess_by_region(
        TRAIN_PATH, TEST_PATH, path_train_std, path_test_std, chunk_size=500_000
    )
    print("Fertig:", stats)
else:
    train = pd.read_csv(TRAIN_PATH)
    test = pd.read_csv(TEST_PATH)
    train["_split"] = "train"
    test["_split"] = "test"
    test["score"] = np.nan
    panel = build_features(pd.concat([train, test], ignore_index=True))
    panel = add_persistence_baseline(panel, lag_days=7)

    train_labeled = panel[(panel["_split"] == "train") & panel["score"].notna()]
    test_feat = panel[panel["_split"] == "test"]
    meta = ["region_id", "date", "year", "month", "day", "ordinal", "score", "score_persist7"]
    save_train = [c for c in list(dict.fromkeys(meta + feat_cols)) if c in train_labeled.columns]
    save_test = [c for c in list(dict.fromkeys(
        ["region_id", "date", "year", "month", "day", "ordinal"] + feat_cols
    )) if c in test_feat.columns]
    train_labeled[save_train].to_parquet(path_train_std, index=False)
    test_feat[save_test].to_parquet(path_test_std, index=False)
    print(f"Gespeichert: {len(train_labeled):,} train | {len(test_feat):,} test")

In [ ]:
import pyarrow.parquet as pq

n_train = pq.read_metadata(path_train_std).num_rows
n_test = pq.read_metadata(path_test_std).num_rows
print(f"Parquet: train_labeled {n_train:,} | test {n_test:,}")
pd.read_parquet(path_train_std).head(3)